In [ ]:
"""
exp01 Step 03: Walk-Forward Evaluation (PROTOCOL.md §9-§12)

Per fold (3-month windows, 10-session embargo):
    1. VALIDATION: for each training window L in {3, 5, 10} years, fit one LightGBM P(TP) model per delta on the sampled
       training rows, decide at EVERY eligible minute of the validation quarter (best expected net return across deltas,
       buy above a threshold), and simulate each (L, threshold) candidate.
    2. SELECTION: validation total return of the quarter (ties: shorter window, higher threshold).
    3. HONEST REPORTING (every run mode, validation data only):
         - every candidate chained over the validation quarters (after the fact: optimistic for the best one);
         - the PRIOR-ONLY path: the setting selected at fold f - 1, on fold f's validation quarter (its decisions of that
           quarter), together with RANDOM_BASELINE_RUN_COUNT random-entry runs of matched activity and the fixed-time entry;
         - time invested (exposure) and the zero-skill reference: a strategy without skill that is invested a fraction f of
           the time is expected to grow about (1 + buy-and-hold)^f - 1;
         - the minimum detectable effect of the quarterly excess returns.
       exp01 positions can extend past the end of a quarter (up to 10 sessions), so the path is summarized per quarter
       (no daily chaining or daily bootstrap here).
    4. TEST (run modes "latest" / "history" only): refit and evaluate the test window ONCE, with the same baselines.

RUN_MODE: "validation_only" (use until the design is frozen), "latest", "history".
Model and decision policy: one LightGBM classifier per delta + expected-return rule (walk_forward.py), a proposal that was
never formally agreed (PROTOCOL.md §10).
Prerequisite: exp01 step 01. Outputs: store04_experiments/exp01_minute_entry/step03_walk_forward_data/.
Run time: about 1 minute per fold for the validation candidates, plus the prior-only baselines (about 45 folds).
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT DATA LOADING AND EXECUTION FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import read_model_dataset_pdf, get_model_feature_col_str_list
from experiments.exp01_minute_entry.walk_forward import get_fold_pdf, run_walk_forward_fold_dict, run_window_with_baseline_dict
# IMPORT THE EVALUATION FUNCTIONS
from so.core.evaluation import get_pooled_selection_pdf, get_prior_only_pdf, get_candidate_summary_pdf, get_window_return_summary_dict, get_baseline_summary_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning
# IMPORT PLOT FUNCTIONS
from so.features.plot_ohlcv_utils import plot_transaction_ohlcv

In [ ]:
# DEFINE THE RUN MODE ("validation_only", "latest" or "history"): NEVER "latest" / "history" BEFORE THE DESIGN IS FROZEN
RUN_MODE_STR = "validation_only"
# DEFINE WHETHER THE TEST WINDOWS ARE EVALUATED
RUN_TEST_BOOL = RUN_MODE_STR in ["latest", "history"]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tTest Windows Evaluated:\t{RUN_TEST_BOOL}")
print(f"Deltas:\t{exp_config.MODEL_DELTA_LIST}")
print(f"Training Windows:\t{exp_config.TRAIN_WINDOW_YEARS_LIST} years\tThresholds:\t{exp_config.EV_THRESHOLD_LIST}")
print(f"Sampling:\t{exp_config.SAMPLING_MODE} ({exp_config.SAMPLE_EVERY_N_MINUTES} min)\tWeights:\t{exp_config.SAMPLE_WEIGHT_MODE}\tEmbargo:\t{exp_config.EMBARGO_TRADING_DAYS} sessions")

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step03_walk_forward_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

In [ ]:
# CALL FUNCTION TO READ THE COMPLETE MINUTE BARS (RAW FOLDER OF so.paths)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS (USED BY THE SIMULATOR AND THE SAMPLE WEIGHTS)
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# CALL FUNCTION TO GET THE WALK-FORWARD FOLDS
fold_pdf = get_fold_pdf(ohlcv_array_dict["session_date_list"])
# SELECT THE FOLDS OF THE RUN MODE
selected_fold_pdf = fold_pdf.iloc[[-1]] if RUN_MODE_STR == "latest" else fold_pdf
# SAVE THE SCHEDULE
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}\tmax timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
print(f"Fold Count:\t{len(fold_pdf)}\tSelected:\t{len(selected_fold_pdf)}")
# PREVIEW DATAFRAME
selected_fold_pdf

In [ ]:
# COLLECT THE EARLIEST TRAINING START AND THE LATEST REFIT END OF THE SELECTED FOLDS
pool_start_date = min(selected_fold_pdf[f"train_start_{train_years}y"].min() for train_years in exp_config.TRAIN_WINDOW_YEARS_LIST)
pool_end_date = selected_fold_pdf["refit_end"].max()
# CALL FUNCTION TO READ THE SAMPLED TRAINING POOL (EVERY TRAINING / REFIT WINDOW IS A SLICE OF IT)
train_pool_pdf = read_model_dataset_pdf(str(pool_start_date), str(pool_end_date), exp_config.MODEL_DELTA_LIST, exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES)
# DEFINE THE MODEL FEATURES
feature_col_str_list = get_model_feature_col_str_list(train_pool_pdf)
# DISPLAY INFORMATION
print(f"Training Pool:\t{pool_start_date} -> {pool_end_date}\t{len(train_pool_pdf):,} rows")
print(f"Model Features ({len(feature_col_str_list)}):\t{feature_col_str_list}")

In [ ]:
# LISTS TO HOLD THE CANDIDATES, THE PRIOR-ONLY PATH AND THE TEST SUMMARIES
candidate_pdf_list, prior_row_dict_list, prior_baseline_pdf_list, fold_summary_dict_list = [], [], [], []
# DEFINE THE SETTING SELECTED AT THE PREVIOUS FOLD (NONE BEFORE THE FIRST FOLD)
previous_best_setting_dict = None
# ITERATE OVER THE SELECTED FOLDS
for _, fold_row in selected_fold_pdf.iterrows():
    # DEFINE THE FOLD ID AND START TIME
    fold_id, start_time = int(fold_row["fold_id"]), time.time()
    # DISPLAY INFORMATION
    print(f"\nFold {fold_id}:\tvalidation {fold_row['valid_start']} -> {fold_row['valid_end']}\ttest {fold_row['test_start']} -> {fold_row['test_end']}")
    # CALL FUNCTION TO READ THE UNSAMPLED VALIDATION AND TEST ROWS (EVERY ELIGIBLE DECISION, AS A LIVE MODEL)
    eval_pool_pdf = read_model_dataset_pdf(str(fold_row["valid_start"]), str(fold_row["test_end"]), exp_config.MODEL_DELTA_LIST, "all", alert_in=False)
    # CALL FUNCTION TO RUN THE FOLD
    fold_result_dict = run_walk_forward_fold_dict(fold_row, train_pool_pdf, eval_pool_pdf, ohlcv_array_dict, feature_col_str_list, run_test_bool_in=RUN_TEST_BOOL)
    fold_candidate_pdf = fold_result_dict["selection_pdf"]
    candidate_pdf_list.append(fold_candidate_pdf)
    # LOG EVERY VALIDATION CANDIDATE (ONE WRITE PER FOLD) AND SAVE THEM
    log_trial_pdf(exp_config, "validation", fold_candidate_pdf, ["train_years", "ev_threshold"], [col for col in fold_candidate_pdf.columns if col.startswith("valid_")], False, fold_pdf)
    write_csv_file_to_path(fold_candidate_pdf, f"{output_path_str}fold{fold_id:03d}_validation_candidate_data.csv", "W", alert_in=False)
    # PRIOR-ONLY PATH: THE SETTING SELECTED AT THE PREVIOUS FOLD, ON THIS FOLD'S VALIDATION QUARTER, WITH ITS BASELINES
    if previous_best_setting_dict is not None:
        decision_pdf = fold_result_dict["valid_decision_pdf_dict"][(previous_best_setting_dict["train_years"], previous_best_setting_dict["ev_threshold"])]
        window_dict = run_window_with_baseline_dict(decision_pdf, fold_result_dict["valid_pdf"], ohlcv_array_dict, fold_row["valid_start"], fold_row["valid_end"])
        prior_row_dict_list.append({"fold_id": fold_id, "selected_at_fold_id": fold_id - 1, **previous_best_setting_dict,
                                    **{f"valid_{key}": value for key, value in window_dict["metric_dict"].items()},
                                    "valid_buy_hold_total_return": window_dict["metric_dict"]["buy_hold_return"]})
        prior_baseline_pdf_list.append(window_dict["baseline_pdf"].assign(fold_id=fold_id))
        print(f"\tPrior-only ({previous_best_setting_dict}):\t{window_dict['metric_dict']['total_return']:.2%} vs buy & hold {window_dict['metric_dict']['buy_hold_return']:.2%}")
    # REMEMBER THIS FOLD'S SELECTION FOR THE NEXT FOLD
    previous_best_setting_dict = fold_result_dict["best_setting_dict"]
    # IF THE TEST WINDOW WAS EVALUATED
    if RUN_TEST_BOOL:
        # LOG THE TEST EVALUATION
        log_trial_dict(exp_config, "test", fold_result_dict["best_setting_dict"], fold_result_dict["test_metric_dict"], True, fold_row)
        # SAVE THE TEST OUTPUTS
        write_csv_file_to_path(fold_result_dict["baseline_metric_pdf"], f"{output_path_str}fold{fold_id:03d}_test_baseline_data.csv", "W", alert_in=False)
        write_csv_file_to_path(fold_result_dict["cost_sensitivity_pdf"], f"{output_path_str}fold{fold_id:03d}_test_cost_sensitivity_data.csv", "W", alert_in=False)
        write_csv_file_to_path(fold_result_dict["test_transaction_pdf"], f"{output_path_str}fold{fold_id:03d}_test_transaction_data.csv", "W", alert_in=False)
        write_csv_file_to_path(fold_result_dict["test_daily_equity_pdf"], f"{output_path_str}fold{fold_id:03d}_test_daily_equity_data.csv", "W", alert_in=False)
        # COLLECT THE RANDOM BASELINE RETURNS AND DEFINE THE FOLD SUMMARY
        baseline_pdf = fold_result_dict["baseline_metric_pdf"]
        random_return_arr = baseline_pdf[baseline_pdf["baseline"].str.startswith("random_")]["total_return"].to_numpy()
        fold_summary_dict_list.append({"fold_id": fold_id, "test_start": fold_row["test_start"], "test_end": fold_row["test_end"],
                                       **fold_result_dict["best_setting_dict"], **fold_result_dict["test_metric_dict"],
                                       "random_mean_return": random_return_arr.mean() if len(random_return_arr) else np.nan,
                                       "model_vs_random_percentile": (random_return_arr < fold_result_dict["test_metric_dict"]["total_return"]).mean() if len(random_return_arr) else np.nan,
                                       "fixed_time_return": baseline_pdf.loc[baseline_pdf["baseline"] == "fixed_time", "total_return"].iloc[0]})
    # DISPLAY INFORMATION
    print(f"\t⏱ Fold {fold_id} processed in {(time.time() - start_time) / 60:.1f} minutes")
# CONCATENATE THE CANDIDATES AND SAVE THEM
candidate_pdf = pd.concat(candidate_pdf_list, ignore_index=True)
write_csv_file_to_path(candidate_pdf, f"{output_path_str}validation_candidate_data.csv", "W")

In [ ]:
"""
After-the-fact view (OPTIMISTIC for the best setting: it is the best of 12 chosen with hindsight). valid_exposure_pct is
the share of the window's bars with an open position. Buy-and-hold is measured per candidate over the window start ->
later of the window end and its last exit, so chained_buy_hold_return differs slightly per candidate.
"""
# CALL FUNCTION TO SUMMARIZE EVERY CANDIDATE OVER THE VALIDATION QUARTERS
candidate_summary_pdf = get_candidate_summary_pdf(candidate_pdf, ["train_years", "ev_threshold"], ["valid_exposure_pct", "valid_trade_count", "valid_skipped_decision_count"])
# ADD THE ZERO-SKILL REFERENCE: (1 + BUY-AND-HOLD) ^ EXPOSURE - 1 (A STRATEGY WITHOUT SKILL, INVESTED THAT SHARE OF THE TIME)
candidate_summary_pdf["zero_skill_reference_return"] = (1 + candidate_summary_pdf["chained_buy_hold_return"]) ** candidate_summary_pdf["mean_valid_exposure_pct"] - 1
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# PREVIEW DATAFRAME
candidate_summary_pdf.round(4)

In [ ]:
"""
Prior-only path (the honest validation estimate) and the information test: the setting selected at fold f - 1 on fold f's
validation quarter, against random entries of matched activity and the fixed-time entry. No test window is read.
"""
# IF THE PRIOR-ONLY PATH HAS ROWS (NOT IN "latest" MODE)
if prior_row_dict_list:
    # CONVERT THE PATH AND THE BASELINES AND SAVE THEM
    prior_only_pdf = pd.DataFrame(prior_row_dict_list)
    prior_baseline_pdf = pd.concat(prior_baseline_pdf_list, ignore_index=True)
    write_csv_file_to_path(prior_only_pdf, f"{output_path_str}prior_only_data.csv", "W", alert_in=False)
    write_csv_file_to_path(prior_baseline_pdf, f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
    # CROSS-CHECK THE PATH WITH THE GENERIC SELECTION (SINGLE QUARTER, TIES: SHORTER WINDOW, HIGHER THRESHOLD)
    generic_prior_pdf = get_prior_only_pdf(candidate_pdf, get_pooled_selection_pdf(candidate_pdf, ["train_years", "ev_threshold"], "valid_total_return",
                                                                                   exp_config.PRIOR_SELECTION_QUARTER_COUNT, [("train_years", True), ("ev_threshold", False)]),
                                           ["train_years", "ev_threshold"])
    assert generic_prior_pdf[["fold_id", "train_years", "ev_threshold"]].astype(float).equals(prior_only_pdf[["fold_id", "train_years", "ev_threshold"]].astype(float))
    # SUMMARIZE THE PATH (QUARTERS) AND THE BASELINES (INFORMATION TEST)
    prior_summary_dict = get_window_return_summary_dict(prior_only_pdf["valid_total_return"].to_numpy(), prior_only_pdf["valid_buy_hold_total_return"].to_numpy())
    baseline_summary_pdf = get_baseline_summary_pdf(prior_baseline_pdf)
    mean_exposure = float(prior_only_pdf["valid_exposure_pct"].mean())
    zero_skill_reference = (1 + prior_summary_dict["chained_buy_hold_return"]) ** mean_exposure - 1
    # DISPLAY INFORMATION
    print(f"Prior-only path ({prior_summary_dict['window_count']} quarters):\t{prior_summary_dict['chained_return']:.2%} vs buy & hold {prior_summary_dict['chained_buy_hold_return']:.2%}"
          f"\t(beats buy & hold in {prior_summary_dict['windows_beating_buy_hold']} quarters, one-sided sign test p = {prior_summary_dict['sign_test_p_value']:.3f})")
    print(f"Mean excess per quarter:\t{prior_summary_dict['mean_excess_return']:.2%} (SD {prior_summary_dict['std_excess_return']:.2%}, t = {prior_summary_dict['t_stat']:.2f})"
          f"\tMinimum detectable effect:\t{prior_summary_dict['mde_per_window']:.2%} per quarter, about {prior_summary_dict['mde_per_year']:.1%} per year")
    print(f"Mean exposure:\t{mean_exposure:.1%}\tZero-skill reference at that exposure:\t{zero_skill_reference:.2%}")
    print(f"PRIMARY on the prior-only path (chained quarters above buy & hold):\t{prior_summary_dict['chained_return'] > prior_summary_dict['chained_buy_hold_return']}")
    # PREVIEW THE INFORMATION TEST
    display(baseline_summary_pdf.round(4))

In [ ]:
# IF THE PRIOR-ONLY PATH HAS ROWS
if prior_row_dict_list:
    # CREATE A FIGURE OF THE PRIOR-ONLY PATH, BUY AND HOLD AND THE RANDOM MEDIAN (CHAINED QUARTERS)
    random_median_pdf = prior_baseline_pdf[prior_baseline_pdf["baseline"].str.startswith("random_")].groupby("fold_id")["total_return"].median()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=prior_only_pdf["fold_id"], y=np.cumprod(1 + prior_only_pdf["valid_total_return"]), mode="lines+markers", name="Model (prior-only)"))
    fig.add_trace(go.Scatter(x=prior_only_pdf["fold_id"], y=np.cumprod(1 + prior_only_pdf["valid_buy_hold_total_return"]), mode="lines+markers", name="Buy & Hold"))
    fig.add_trace(go.Scatter(x=random_median_pdf.index, y=np.cumprod(1 + random_median_pdf.to_numpy()), mode="lines+markers", name="Random entry (median per quarter)"))
    fig.update_layout(title="Prior-Only Path On Validation Quarters (growth of 1)", template="plotly_dark", width=1100, height=500, font=dict(color="white"))
    fig.show()
    # LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATES ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
    summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": "previous quarter", "path": "prior-only on validation"},
                                        {**prior_summary_dict, "mean_exposure": mean_exposure, "zero_skill_reference_return": zero_skill_reference,
                                         "baseline_chained_return_dict": {row["baseline"]: row["chained_return"] for _, row in baseline_summary_pdf.iterrows()},
                                         "best_after_the_fact_chained_return": float(candidate_summary_pdf["chained_return"].iloc[0])},
                                        False, note_str_in="exp01 step03 validation summary (prior-only path, baselines, exposure, MDE)")

In [ ]:
# CONVERT THE TEST FOLD SUMMARIES TO A DATAFRAME
fold_summary_pdf = pd.DataFrame(fold_summary_dict_list)
# IF THERE ARE TEST RESULTS
if not fold_summary_pdf.empty:
    # SAVE THE FOLD SUMMARY
    write_csv_file_to_path(fold_summary_pdf, f"{output_path_str}fold_summary_data.csv", "W")
    # SUMMARIZE THE TEST WINDOWS
    test_summary_dict = get_window_return_summary_dict(fold_summary_pdf["total_return"].to_numpy(), fold_summary_pdf["buy_hold_return"].to_numpy())
    # DISPLAY INFORMATION
    print(f"Folds Beating Buy & Hold:\t{test_summary_dict['windows_beating_buy_hold']}/{test_summary_dict['window_count']}\t(one-sided sign test p-value: {test_summary_dict['sign_test_p_value']:.4f})")
    print(f"Chained Model Return:\t\t{test_summary_dict['chained_return']:.2%}")
    print(f"Chained Buy & Hold Return:\t{test_summary_dict['chained_buy_hold_return']:.2%}")
    print(f"Mean Sharpe (model / B&H):\t{fold_summary_pdf['sharpe_ratio'].mean():.2f} / {fold_summary_pdf['buy_hold_sharpe_ratio'].mean():.2f}")
    print(f"Worst Drawdown (model / B&H):\t{fold_summary_pdf['max_drawdown'].min():.2%} / {fold_summary_pdf['buy_hold_max_drawdown'].min():.2%}")
    print(f"Total Trades:\t\t\t{int(fold_summary_pdf['trade_count'].sum()):,}")
# PREVIEW DATAFRAME
fold_summary_pdf

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)